# GDELT feature builder (Stream A)

**One job:** turn raw chip events into a per-country-per-year risk table and **save it** to `gdelt_features_by_country_year.parquet`.
The GAT notebook then just loads that file.

In [1]:
import pandas as pd
import numpy as np
from country_codes import ISO3_2_M49

## Step 1 — scorer + aggregation functions
(run once; defines everything)

In [2]:
CRITICAL_ROOTS = ["14", "17", "18", "19", "20"]
ROOT_WEIGHTS   = {"14": 1.0, "17": 1.5, "18": 2.0, "19": 3.0, "20": 4.0}  # fight > protest

def _num(s): return pd.to_numeric(s, errors="coerce")

# ---------- pluggable per-event scorers ----------
def score_severity_salience(d):
    """|Goldstein| x NumMentions : type-severity scaled by how loudly it was reported."""
    return _num(d["GoldsteinScale"]).abs() * _num(d["NumMentions"])

def score_reliability_weighted(d):
    """+ reward corroboration across distinct sources."""
    return score_severity_salience(d) * np.log1p(_num(d["NumSources"]))

def score_root_weighted(d):
    """+ weight by event type (fight/mass-violence outweigh protest)."""
    w = d["EventRootCode"].astype(str).map(ROOT_WEIGHTS).fillna(1.0)
    return score_severity_salience(d) * w

# ---------- country-month aggregation (normalized) ----------
def score_to_country_month(ev, scorer=score_severity_salience,
                           roots=CRITICAL_ROOTS, country_col="Actor1CountryCode"):
    d = ev.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.dropna(subset=["date", country_col])
    if roots is not None:
        d = d[d["EventRootCode"].astype(str).isin(roots)]
    d["ym"]    = d["date"].dt.to_period("M")
    d["score"] = scorer(d)
    d["gold"], d["ment"], d["tone"] = (_num(d["GoldsteinScale"]),
                                       _num(d["NumMentions"]), _num(d["AvgTone"]))
    d["gm"] = d["gold"] * d["ment"].clip(lower=1)
    g = d.groupby([country_col, "ym"])
    out = g.agg(
        n_events   = ("GLOBALEVENTID", "size"),
        score_mean = ("score", "mean"),
        score_max  = ("score", "max"),
        score_sum  = ("score", "sum"),
        tone_mean  = ("tone", "mean"),
        gm_sum     = ("gm", "sum"),
        ment_sum   = ("ment", "sum"),
    )
    out["goldstein_wmean"] = out["gm_sum"] / out["ment_sum"].clip(lower=1)
    return (out.drop(columns=["gm_sum", "ment_sum"]).reset_index()
               .rename(columns={country_col: "iso3"}))

def score_month_to_annual(cm):
    cm = cm.copy(); cm["year"] = cm["ym"].dt.year
    g = cm.groupby(["iso3", "year"])
    return g.agg(
        events_total    = ("n_events", "sum"),
        score_mean      = ("score_mean", "mean"),
        score_max       = ("score_max", "max"),
        score_vol       = ("score_mean", "std"),
        goldstein_wmean = ("goldstein_wmean", "mean"),
        tone_mean       = ("tone_mean", "mean"),
        active_months   = ("n_events", lambda s: (s > 0).sum()),
    ).reset_index()

## Step 2 — build the country-year table and SAVE it
This is the file the GAT notebook loads.

In [3]:
iso2m49 = {**ISO3_2_M49, "TWN": "490"}   # Taiwan -> Comtrade 490

ev = pd.read_parquet("gdelt_chip_events_2017_2023.parquet")
ev = ev[pd.to_datetime(ev["date"]).dt.year.between(2017, 2023)]

cm = score_to_country_month(ev, scorer=score_severity_salience)   # -> country x month
cy = score_month_to_annual(cm)                                    # -> country x year

cy["reporterCode"] = cy["iso3"].map(iso2m49)
cy = cy.dropna(subset=["reporterCode"])
cy["reporterCode"] = cy["reporterCode"].astype(int)

cy.to_parquet("gdelt_features_by_country_year.parquet", index=False)
print("SAVED:", cy.shape, "-> gdelt_features_by_country_year.parquet")
print("years:", sorted(cy["year"].unique()))
cy.head(12)

SAVED: (1272, 10) -> gdelt_features_by_country_year.parquet
years: [2017, 2018, 2019, 2020, 2021, 2022, 2023]


,iso3,year,events_total,score_mean,score_max,score_vol,goldstein_wmean,tone_mean,active_months,reporterCode
0,ABW,2017,6,63.333333,110.0,NaN,-5.000000,-7.958112,1,533
1,ABW,2018,5,261.000000,774.0,254.558441,-9.000000,-9.552747,2,533
2,ABW,2019,1,40.000000,40.0,NaN,-5.000000,-0.731296,1,533
3,ABW,2020,1,550.000000,550.0,NaN,-5.000000,0.649351,1,533
4,ABW,2023,3,306.250000,510.0,47.729708,-7.500000,-7.038944,2,533
5,AFG,2017,1224,413.157917,16280.0,168.261722,-9.346548,-5.113536,12,4
6,AFG,2018,1713,536.131702,20940.0,207.623912,-9.593480,-4.995118,12,4
7,AFG,2019,1521,535.003495,24813.0,102.936753,-9.528049,-3.932267,12,4
8,AFG,2020,551,240.283823,3258.0,77.333322,-9.144026,-4.222905,12,4
9,AFG,2021,1523,329.279303,23490.0,141.958074,-9.156032,-4.027143,12,4


## Optional — inspect raw events (viewer, not part of the pipeline)

In [4]:
def _as_set(v):
    if v is None: return None
    if isinstance(v, (list, tuple, set, range)): return set(v)
    return {v}

def events_in(ev, country=None, year=None, month=None,
              country_field="any", sort_by="intensity", top=None):
    d = ev.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    if country is not None:
        cs = _as_set(country)
        if country_field == "any":
            mask = (d["Actor1CountryCode"].isin(cs) |
                    d["Actor2CountryCode"].isin(cs) |
                    d["ActionGeo_CountryCode"].isin(cs))
        else:
            mask = d[country_field].isin(cs)
        d = d[mask]
    ys, ms = _as_set(year), _as_set(month)
    if ys is not None: d = d[d["date"].dt.year.isin(ys)]
    if ms is not None: d = d[d["date"].dt.month.isin(ms)]
    if d.empty:
        print("No events for that selection"); return d
    for c in ["GoldsteinScale", "NumMentions"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["intensity"] = d["GoldsteinScale"].abs() * d["NumMentions"]
    d = d.sort_values(sort_by, ascending=(sort_by == "date"))
    return d.head(top) if top else d

# example:
# ev = pd.read_parquet("gdelt_chip_events_2017_2023.parquet")
# events_in(ev, "KOR", 2019, top=10)

---


In [5]:
def score_to_country_month_topk(ev, scorer=score_severity_salience,
                                roots=CRITICAL_ROOTS, country_col="Actor1CountryCode",
                                k=10):
    """Keep only the k highest-scored events per country-month, then aggregate."""
    d = ev.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.dropna(subset=["date", country_col])
    if roots is not None:
        d = d[d["EventRootCode"].astype(str).isin(roots)]
    d["ym"]    = d["date"].dt.to_period("M")
    d["score"] = scorer(d)

    # --- the only new step: keep top-k per country-month ---
    d = (d.sort_values("score", ascending=False)
           .groupby([country_col, "ym"], group_keys=False)
           .head(k))

    # from here, identical to the keep-all aggregation
    d["gold"], d["ment"], d["tone"] = (_num(d["GoldsteinScale"]),
                                       _num(d["NumMentions"]), _num(d["AvgTone"]))
    d["gm"] = d["gold"] * d["ment"].clip(lower=1)
    g = d.groupby([country_col, "ym"])
    out = g.agg(
        n_events   = ("GLOBALEVENTID", "size"),
        score_mean = ("score", "mean"),
        score_max  = ("score", "max"),
        score_sum  = ("score", "sum"),
        tone_mean  = ("tone", "mean"),
        gm_sum     = ("gm", "sum"),
        ment_sum   = ("ment", "sum"),
    )
    out["goldstein_wmean"] = out["gm_sum"] / out["ment_sum"].clip(lower=1)
    return (out.drop(columns=["gm_sum", "ment_sum"]).reset_index()
               .rename(columns={country_col: "iso3"}))

In [6]:
iso2m49 = {**ISO3_2_M49, "TWN": "490"}
ev = pd.read_parquet("gdelt_chip_events_2017_2023.parquet")
ev = ev[pd.to_datetime(ev["date"]).dt.year.between(2017, 2023)]

cm_k = score_to_country_month_topk(ev, k=10)
cy_k = score_month_to_annual(cm_k)          # reuse the SAME annual rollup
cy_k["reporterCode"] = cy_k["iso3"].map(iso2m49)
cy_k = cy_k.dropna(subset=["reporterCode"])
cy_k["reporterCode"] = cy_k["reporterCode"].astype(int)

cy_k.to_parquet("gdelt_features_topk.parquet", index=False)
print("SAVED:", cy_k.shape, "-> gdelt_features_topk.parquet")

SAVED: (1272, 10) -> gdelt_features_topk.parquet
